# 03 — Numbers Inside Computers

**Master syllabus coverage:** V2 1.3

## Why this matters

Precision determines range, error, memory traffic, and hardware throughput. Mixed precision and quantization are numerical policies, not merely smaller file formats.

## Learning objectives

- Explain sign, exponent, significand, range, precision, and machine epsilon.
- Contrast FP16 and BF16 behavior.
- Measure accumulation error and use stable formulas.
- Implement symmetric INT8- and INT4-like quantization and measure reconstruction error.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Finite bits approximate real numbers

Floating point stores sign, exponent, and significand. More exponent bits increase range;
more significand bits increase relative precision. Machine epsilon is the gap between 1
and the next representable value. Overflow produces infinity; underflow produces tiny
subnormal values or zero. Rounding occurs after nearly every operation.


In [ ]:
import math
import numpy as np
import torch

print("0.1 + 0.2 =", format(0.1 + 0.2, ".18f"))
for dtype in (np.float64, np.float32, np.float16):
    info = np.finfo(dtype)
    print(dtype.__name__, "bytes=", np.dtype(dtype).itemsize,
          "epsilon=", info.eps, "tiny=", info.tiny, "max=", info.max)


## 2. Precision and range are separate

FP16 and BF16 both use 16 bits. FP16 has more fraction precision but a much smaller
exponent range; BF16 preserves float32-like range with coarser spacing. FP8 formats make
different exponent/fraction tradeoffs. Integers require an explicit scale and often a
zero point to approximate real values.


In [ ]:
values = torch.tensor([1e-8, 1.0001, 1e4, 1e8], dtype=torch.float32)
for dtype in (torch.float16, torch.bfloat16):
    restored = values.to(dtype).float()
    print(dtype, restored.tolist())
assert torch.isinf(values.to(torch.float16)[-1])
assert torch.isfinite(values.to(torch.bfloat16)[-1])


## 3. Accumulation precision matters

Adding a small value to a much larger running total can round the increment away. Order
changes floating-point sums because arithmetic is not perfectly associative. Pairwise or
compensated summation and higher-precision accumulators reduce error.


In [ ]:
increments = np.full(100_000, 1e-3, dtype=np.float16)
sum_fp16 = np.sum(increments, dtype=np.float16)
sum_fp32 = np.sum(increments, dtype=np.float32)
expected = 100.0
print("FP16 accumulation:", float(sum_fp16), "FP32 accumulation:", float(sum_fp32))
assert abs(float(sum_fp32) - expected) < abs(float(sum_fp16) - expected)

rng = np.random.default_rng(42)
a = rng.normal(size=4096).astype(np.float16)
b = rng.normal(size=4096).astype(np.float16)
reference = np.dot(a.astype(np.float64), b.astype(np.float64))
low = np.sum(a * b, dtype=np.float16)
mixed = np.sum(a * b, dtype=np.float32)
print("dot errors: FP16=", abs(float(low) - reference), "mixed=", abs(float(mixed) - reference))


## 4. Stable formulas are part of correctness

Exponentiation overflows quickly. Softmax subtracts the maximum logit because adding a
constant does not change the mathematical distribution. Similar rewrites stabilize
log-likelihood, norms, variance, and normalization.


In [ ]:
logits = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(logits) / np.exp(logits).sum()
shifted = logits - logits.max()
stable = np.exp(shifted) / np.exp(shifted).sum()
print("naive:", naive, "stable:", stable)
assert np.isnan(naive).any() and np.isfinite(stable).all()


## 5. Quantization maps floats to a finite integer grid

Symmetric signed quantization selects scale $s=max|x|/q_{max}$, stores
$q=\operatorname{clip}(\operatorname{round}(x/s))$, and reconstructs $\hat x=sq$.
INT4 stores 16 levels conceptually; physical packing of two 4-bit values per byte is a
separate systems concern.


In [ ]:
def symmetric_quantize(x: np.ndarray, bits: int):
    qmax = 2 ** (bits - 1) - 1
    scale = np.max(np.abs(x)) / qmax
    q = np.clip(np.round(x / scale), -qmax, qmax).astype(np.int8)
    return q, scale, q.astype(np.float32) * scale

signal = np.linspace(-2.5, 2.5, 1000, dtype=np.float32)
for bits in (8, 4):
    q, scale, reconstructed = symmetric_quantize(signal, bits)
    error = np.sqrt(np.mean((signal - reconstructed) ** 2))
    conceptual_bytes = signal.size * bits / 8
    print(f"INT{bits}: scale={scale:.5f}, RMSE={error:.5f}, conceptual bytes={conceptual_bytes:.0f}")


## Failure modes to recognize

- **Overflow/underflow:** finite mathematical values become infinity or zero.
- **Low-precision accumulation:** small contributions vanish from a large sum.
- **FP16/BF16 equivalence assumption:** range and fraction tradeoffs are ignored.
- **Uncalibrated quantization:** clipping or coarse scales destroy important values.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Decode the conceptual roles of sign, exponent, and significand for one IEEE-754 value.
2. Compare dot-product error across input and accumulation dtype combinations.
3. Quantize a skewed distribution per tensor and per channel and compare error.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can choose storage and accumulation precision while predicting range, rounding, memory, and reconstruction consequences.

**Next:** 04 — Parallelism, kernels, and synchronization.
